____
### 0. Preamble
- This serves as a continuation to my other notebook files where models are trained in a plain environment
- Within this notebook file, I will be experimenting with a world model version of the 
- I will also be documenting my learnings from the paper : https://worldmodels.github.io/

____
### 1. Feedforward network vs Recurrent Neural Networks (RNN)

#### 1.2 Feedforward networks
- All neural networks implemented thus far are all **feedforward network** (`nn.Sequential` of `nn.Linear` layers). 
- **Feedforward networks** have no memory: given an input, they compute an output, and that's it(Nothing about the computation depends on what inputs came before.)
    - Feeding in the same `obs_t` in twice in a row, obtains the exact same output twice — there's no notion of "time" or "history" built into the architecture itself. 
    - This is fine for the PPO/TD3/SAC agents because the environment is already Markovian: 
    - `obs = [inventory, days_left, last_demand]` fully captures everything the policy needs to act optimally at that instant, so there's nothing useful left in the *history* of observations that isn't already summarized in the current one.

#### 1.3 Recurrent Neural Network (RNN)
- **Recurrent Neural Network (RNN)** carries a hidden state $h_t$ forward from one step to the next
    - The hidden state is updated as a function of both the new input and the previous hidden state:
            $$h_t = f(x_t, h_{t-1})$$
    - This means the network's output at time $t$ can depend on everything it has seen since the start of the sequence, not just the current input
    - Feed it the same $x_t$ twice in a row at two different points in a sequence, and you can get two different outputs, because $h_{t-1}$ was different each time.

| | Feedforward (e.g. `PPOActorCritic`) | Recurrent (e.g. `MDNRNN`) |
|---|---|---|
| Input → Output | `y = f(x)` | `y_t = f(x_t, h_{t-1})` |
| Memory | None — stateless | Hidden state $h_t$ carried across steps |
| Good for | Tasks where the current input already contains everything needed (Markovian state) | Tasks where what happened *before* matters — sequences, time series, language |
| In this notebook | The policy/value networks for PPO, TD3, SAC | The MDN-RNN world model (M), which needs to remember *trends* across many steps to predict the future |
    


```python
# Feedforward layer: stateless, same weights applied independently each call
linear = nn.Linear(in_features, out_features)
y = linear(x)                     # y depends only on x

# Recurrent layer: stateful, hidden state threads through calls
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
out, (h, c) = lstm(x, (h_prev, c_prev))   # out depends on x AND h_prev
```

____
### 2. Context for World model

#### 2.1 Components of a World model
- There are 3 components to creating a world model 
    1. Vision Model (V)
        - Meant to encode overly complex environment state features into a single z vector
        - Can be used in reverse to encode z vectors ($z$) back into the complex environment 
        - $z_t$ refers to the z vector at timestep $t$
    2. Predictive Model (M)
        - Takes in input z vectors from V and generates future z vectors that V is expected to produce
        - Essentially predicting the next state of the environment
    3. Controller (C)
        - Reads the z vector and any hidden state vectors to produce an output action 


#### 2.2 Dream Environments
- Dream environments are created using M and decoded using V where necessary 
- How a Dream environment is constructed
    1. Produce a probability distribution of $z_{t+1}$ given a specefic $z_t$
    2. Sample a $z_{t+1}$ and use sample as a real observation
    3. Producing a distribution of $z_{t+1}$ creates a "dream environment"
- Therefore a Dream environment is created using the predictive model (M) which produces a state (represented by $z_t$)
    - predictive model (M) also needs to generate a binary indicator to indicate if the episode is done ($d_t$)
    - ie, M($z_t$, $d_t$) -> $z_{t+1}$, $d_{t+1}$
- Usually no V model is required to decode/encode the z vectors as the controller (C) model is able to read the state (only needs decoding for humans to read)

#### 2.3 Training inside Dreams
- Predictive Model (M) used to simulate RNN is RNN(Recurrent-Neural-Network) based
    - RNNs read a hidden state vector and essentially retains 'memory', generating the next z vector based on the previous states and hidden states (refer to the top)
- Thus RNNs help M to predict the next state ($z_{t+1}$, $d_{t+1}$) properly, mimicing a complete game environment
    - If environment consists of pixels, M will have to learn from only raw image data collected from random episodes
- The environment can be made more challenging for training by adjusting the temperature parameter $T$
    -  This is done by increasing $T$ during the sampling process of $z_{t+1}$

#### 2.4 Exploting Virtual Environment
1. M is only an appropriate probabilistic model of the environment (a probabilistic model that mimics the actual environment)
    - Occassionally generate states that don't follow the laws governing the actual environment
2. Controller is given full access to hidden states of M
    - Since the agent is trained using M, it is given access to the hidden states on top of the observable z vectors
    - Therefore the agent is granted access to all internal states and memory of the game engine instead of ONLY game observations
- Therefore agents can efficiently explore ways to directly manipulate hidden states of teh game engine when training to maximize expected cumulative results
- If model finds an adversarial policy that can fool the dynamics model  
    - The model looks good under virtual environment but performs poorly in an actual environment
    - This is because the model visits states where the model is wrong as they are away from training distribution (game states or states that are not within the regular training distribution)
3. Solution for Exploitation
    - use Mixture density RNN (MDNRNN) which approximates the environment to a stochastic one, even if it is deterministic
        - This allows controller (C) to train inside a stochastic environment
    - Adjust temperature parameter $T$ to control randomness
        - The higher the $T$ the more uncertain the environment is, preventing C from taking advantage of imperfections of world model
        - Higher $T$ also trains the model to handle noise better when thrown into the real environment

____
### 3. Imports

In [1]:
import gymnasium as gym
import numpy as np
import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import copy

____
### 4. Import Original Environment

In [2]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        reward = reward / 100.0
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        if truncated and self.inventory > 0:
            reward -= self.inventory * 2.0  # $2 penalty per unsold unit
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory / self.max_inventory, # 0 to 1
                        (self.max_steps - self.step_count) / self.max_steps, # 0 to 1
                        self.last_demand / self.max_inventory] # 0 to 1
                        , dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 0.8
        noise = np.random.normal(0, 2) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

#### 4.1 General Breakdown of Environment
1. General Stats
    - 30 steps per episode
    - Unit cost of $5
    - Price floor of $5 and price ceiling of $50 
2. Observation State
    - an array of 3 numbers, representing inventory, days left and number of units sold last
    - normalise to a number between 0 to 1
    - In the world model, this would be represented by the z vector with 3 numbers 
    - The V model will just be a simple function that reverses the normlisation
3. Rewards (Profit - Penalty)
    - Profit given by (price - 5) * units sold
    - Penalty given by 2 * units left

____
### 5. Creating Components

#### 5.1 Vision Model (V)
- Implemented as a simple function that reverses the normalisation of the state vector
- No need to implement as a seperate class 

In [3]:
min = np.array([0, 0, 0])
max = np.array([100, 30, 100])

In [ ]:
def decode(state : torch.tensor) -> torch.tensor: # normalising to between ranges 0-1
    return state / torch.tensor(max) 

In [5]:
def encode(normalised: torch.tensor) -> torch.tensor:  # convert to tensor of raw numbers
    return normalised * torch.tensor(max)

#### 5.2 Predictive Model (M) 
- Engine for prediction implemented as a Mixture Density RNN (`class MDNRNN`)

In [ ]:
class MDNRNN(nn.Module):
    def __init__(self, env: gym.env, hidden_size=128, n_gaussians=5):
        super().__init__()
        self.obs_dim = env.observation_space.shape[0]  
        self.action_dim = env.action_space.shape[0]
        self.hidden_size = hidden_size
        self.n_gaussians = n_gaussians
        input_dim = self.obs_dim + self.action_dim
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True) 
        # MDN head: for each of K gaussians, need (pi, mu, sigma) per obs_dim
        K = n_gaussians
        self.pi_head = nn.Linear(hidden_size, K)
        self.mu_head = nn.Linear(hidden_size, K * self.obs_dim)
        self.sigma_head = nn.Linear(hidden_size, K * self.obs_dim)
        self.done_head = nn.Linear(hidden_size, 1) # done predictor: simple Bernoulli logit (paper uses a >50% cutoff rather than sampling, "more stable" than sampling from Bernoulli)
 
    def forward(self, z, a, hidden=None):
        x = torch.cat([z, a], dim=-1)
        out, hidden = self.lstm(x, hidden)  # out: (batch, seq_len, hidden_size)
        K, D = self.n_gaussians, self.obs_dim
        pi = torch.softmax(self.pi_head(out), dim=-1)
        mu = self.mu_head(out).view(*out.shape[:-1], K, D)
        sigma = torch.exp(self.sigma_head(out)).clamp(min=1e-4).view(*out.shape[:-1], K, D)
        done_logit = self.done_head(out)
        return pi, mu, sigma, done_logit, hidden
 
    def sample(self, pi, mu, sigma, temperature=1.0):
        batch = pi.shape[0]
        K, D = mu.shape[1], mu.shape[2]
        logits = torch.log(pi.clamp(min=1e-8)) / temperature # temperature-adjusted mixture weights (softmax with temp) and stds
        pi_t = torch.softmax(logits, dim=-1)
        sigma_t = sigma * np.sqrt(temperature)
        # pick a component per batch row
        comp = torch.multinomial(pi_t, num_samples=1).squeeze(-1)  # (batch,)
        idx = comp.view(batch, 1, 1).expand(batch, 1, D)
        chosen_mu = mu.gather(1, idx).squeeze(1)        # (batch, D)
        chosen_sigma = sigma_t.gather(1, idx).squeeze(1)  # (batch, D)
        eps = torch.randn_like(chosen_mu)
        z_next = chosen_mu + eps * chosen_sigma
        return z_next
 
    @staticmethod
    def mdn_loss(pi, mu, sigma, target):
        target = target.unsqueeze(2)  # (batch, seq_len, 1, D)
        # per-component, per-dim log prob, summed over D (factored gaussian,
        log_prob = -0.5 * (((target - mu) / sigma) ** 2 + 2 * torch.log(sigma) + np.log(2 * np.pi))
        log_prob = log_prob.sum(dim=-1)  # (batch, seq_len, K)
        log_pi = torch.log(pi.clamp(min=1e-8))
        log_mix = torch.logsumexp(log_pi + log_prob, dim=-1)  # (batch, seq_len)
        return -log_mix.mean()
 
    def init_hidden(self, batch_size, device):
        h0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
        return (h0, c0)

##### 5.21 Initialisation
 
```python
def __init__(self, env: gym.env, hidden_size=128, n_gaussians=5):
    super().__init__()
    self.obs_dim = env.observation_space.shape[0]  
    self.action_dim = env.action_space.shape[0]
    self.hidden_size = hidden_size
    self.n_gaussians = n_gaussians 
    input_dim = self.obs_dim + self.action_dim
    self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True) 
```
- `env : gym.env` is the environment the MDNRNN is trying to mimic
- `self.obs_dim = env.observation_space.shape[0]` , `self.action_dim = env.action_space.shape[0]` : the observation dimenstion and action dimension of the action 
- `self.hidden_size = hidden_size` : refers to the size of the hidden state vector
    - The entire history of an episode gets squeezed into a vector of 128 numbers by default
    - The larger the side of the hidden state vector, the larger the model's capacity to represent complex temporal patterns, but more parameters to train, slower to train, and easier to overfit on a small dataset of rollouts.
- `self.n_gaussians = n_gaussians` : the number of gaussian components that make up the predicted mixture distribution for $z_{t+1}$
- `self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True)` : a Long Short Term Memory (LSTM) is a specefic type of RNN

```python
K = n_gaussians
self.pi_head = nn.Linear(hidden_size, K)
self.mu_head = nn.Linear(hidden_size, K * obs_dim)
self.sigma_head = nn.Linear(hidden_size, K * obs_dim)
self.done_head = nn.Linear(hidden_size, 1)
```
- `K = n_gaussians` : stores the number of gaussian components as K
- `self.pi_head = nn.Linear(hidden_size, K)` : a Linear model that transforms the hidden state into K values, each number in the vector representing a weight
    - each weight represents the probability of the next state landing near that cluster
- `self.mu_head = nn.Linear(hidden_size, K * obs_dim)`: a Linear layer that transforms the hidden state into a vector of `K * obs_dim` values (applied independently at every batch/timestep position)
    - every `obs_dim`-sized chunk of that output represents one of the K candidate next-states
    - each number within that chunk is the predicted mean for one dimension of the observation (inventory, days_left, or last_demand)
    - this effectively generates K mean vectors, each of size `obs_dim`, one per gaussian component in the mixture
- `self.sigma_head = nn.Linear(hidden_size, K * obs_dim)` : a Linear layer that transforms the hidden state into a vector of `K * obs_dim` values (applied independently at every batch/timestep position)
    - same explanation just that each value represents the standard deviation of the corresponding mean value

- Therefore, instead of predicting a single gaussian (one mean, one spread) for the next state, the MDN predicts K separate gaussians plus a weight (pi) for each, saying "the next state could land near this cluster with this probability, or near that cluster with that probability." 
    - calling `sample()`, picks one of the K components (weighted by pi) and then samples from that specific gaussian.

- `self.done_head = nn.Linear(hidden_size, 1)` : a Linear layer that transforms the hidden state into a single binary value, representing whether the episode is complete


##### 5.22 `forward()`
- Receives a state and action and returns gaussian components and state indicator
- State `z` and action `a`, both in the form of tensors 
    - `z` is of the shape `(batch, seq_len, obs_dim)`
    - `a` is of the shape `(batch, seq_len, actionn_dim)`
- Gaussian components : `pi`, `mu`, `sigma`
    - State indicator : `done_logit`

``` python
def forward(self, z, a, hidden=None):
    x = torch.cat([z, a], dim=-1)
    out, hidden = self.lstm(x, hidden)
```
- `torch.cat([z, a], dim=-1)`: glues the state vector and action vector together side by side, so the LSTM sees them as one combined input at each timestep. 
    - If `z = (batch, seq_len, 3)` and `a = (batch, seq_len, 1)`, `x = (batch, seq_len, 4).`
- `out, hidden = self.lstm(x, hidden)` : feeds x and hidden into the LSTM over the whole sequence.
    - hidden is the hidden state from before this sequence started
    - default value of `hidden = None` since the first call does not have any hidden states yet  
    - The LSTM internally steps through each timestep, updating its hidden state as it goes, and returns `out` and `hidden`
    - `out` represents the hidden state at every timestep ((batch, seq_len, hidden_size)) 
    - `hidden` represents the final hidden state after the last timestep in the sequence.

```python
K, D = self.n_gaussians, self.obs_dim
pi = torch.softmax(self.pi_head(out), dim=-1)
mu = self.mu_head(out).view(*out.shape[:-1], K, D)
sigma = torch.exp(self.sigma_head(out)).clamp(min=1e-4).view(*out.shape[:-1], K, D)
done_logit = self.done_head(out)
```
- `pi = torch.softmax(self.pi_head(out), dim=-1)` : reads from `out` and returns a vector with K values 
    - `torch.softmax()` is used to guarantee that the numbers inside the vector obtained are all positive and add up to 1 since the values in pi represents weights and are all probabilities
- `mu = self.mu_head(out).view(*out.shape[:-1], K, D)` : reads from `out` and returns an output with K components and D dimensions per component
    - `.view(*out.shape[:-1], K, D)` is used to view the values without constraining the values
    - the parameters `K, D` makes the output easily indexable : Kth component, D dimensions
- `sigma = torch.exp(self.sigma_head(out)).clamp(min=1e-4).view(*out.shape[:-1], K, D)` : reads from `out` and returns an output with K components and D dimensions per component
    - `torch.exp()` is used to make the standard deviation +ve since $e^x$ > 0 for all real values of x
    - `.clamp(min=1e-4)` adds a hard floor on the values so it never gets numerically close to zero
    - `.view(*out.shape[:-1], K, D)` is used to view the values without constraining the values
    - the parameters `K, D` makes the output easily indexable : Kth component, D dimensions
- `done_logit = self.done_head(out)` : reads from `out` and returns an output
    - no other functions needed as it is just a raw product

```python
return pi, mu, sigma, done_logit, hidden
```
- pi: `(batch, seq_len, K)`
- mu: `(batch, seq_len, K, obs_dim)`
- sigma: `(batch, seq_len, K, obs_dim)`
- done_logit: `(batch, seq_len, 1)`

##### 5.23 `sample()`
- Receives gaussian components and a temperature parameter and returns the next state vector
- Gaussian components: `pi`, `mu`, `sigma` which makes up the distribution
    - pi : `(batch, seq_len, K)`
    - mu: `(batch, seq_len, K, obs_dim)`
    - sigma: `(batch, seq_len, K, obs_dim)`
- Temperature parameter is used to control the randomness of the sampling and determines how unpredictable the general environment is
    - applied to `pi` : increases / decreases the differences in logit, making the selection of gaussian more random 
    - applied to `sigma` : increases / decreases the standard deviation of every gaussian, increasing/decreasing noise
- Turns distributions into one concrete z_{t+1} that can feed back into the environment loop.

```python
def sample(self, pi, mu, sigma, temperature=1.0):
    batch = pi.shape[0]
    K, D = mu.shape[1], mu.shape[2]
```
- `batch = pi.shape[0]` and `K, D = mu.shape[1], mu.shape[2]` : reads off the dimension size from the tensors so it is not hardcoded for the rest of the function

```python
logits = torch.log(pi.clamp(min=1e-8)) / temperature
pi_t = torch.softmax(logits, dim=-1)
sigma_t = sigma * np.sqrt(temperature)
```
- `torch.log(pi.clamp(min=1e-8)) / temperature` : logs the values within the pi tensor and implements a floor of `1e-8`  and divides it by temperature
    - applying `log()` converts it to logit space 
    - dividing by temperature stretches/compresses the logits 
- `pi_t = torch.softmax(logits, dim=-1)` : applying `softmax()` converts the whole logit tensor back into proper probability so that the all the numbers add up to 1
    - temperature < 1 : dividing by a fraction amplifies the logit differences, making `pi_t` more peaked , making the already more likely components more dominant (less randomness since more obvious)
    - temperature > 1 : flattens the distribution towards uniform, components become equally more likely and more ranndomness in which one gets picked (more ranndomness)
- `sigma_t = sigma * np.sqrt(temperature)` : seperately scales how spread out each individual gaussian is 
    - temperature > 1: sigma increases, distribution becomes wider, more noise
    - temperature < 1: sigma decreases, distribution becomes narrower, less noise

```python
comp = torch.multinomial(pi_t, num_samples=1).squeeze(-1)  # (batch,)
idx = comp.view(batch, 1, 1).expand(batch, 1, D)
chosen_mu = mu.gather(1, idx).squeeze(1)        # (batch, D)
chosen_sigma = sigma_t.gather(1, idx).squeeze(1)  # (batch, D)
eps = torch.randn_like(chosen_mu)
z_next = chosen_mu + eps * chosen_sigma
return z_next
```

- `comp = torch.multinomial(pi_t, num_samples=1).squeeze(-1)` : chooses the exact gaussian to use based on the `pi_t` tensor that stores the processed probability of all gaussians 
    - `torch.multinomial()` treats each row of `pi_t` as a probability distribution over K outcomes and draws one random index per batch row, weighted by those probabilities.
    - `.squeeze(-1)` removes a redundant size-1 dimension multinomial leaves behind, squeezing it from `(batch, 1)` to just `(batch,)`
    - therefore `comp` is just a flat list of chosen indices, one per batch row with shape (batch,)
- `idx = comp.view(batch, 1, 1).expand(batch, 1, D)` : reshapes the chosen indices
    - `.view(batch, 1, 1)` turns comp into the shape `(batch, 1, 1)`
    - `.expand(batch, 1, D)` repeats that single index across all D positions to indicate that the same index should be used to choose the mean and standard deviaton to obtain the distribution to be used
- `chosen_mu = mu.gather(1, idx).squeeze(1)` : chooses the `mu` (mean) to use
    - `.gather(1, idx)` picks out, along dimension 1 (the K dimension)
    -  i.e., "for batch row 0, give me mu[0, comp[0], :]; for batch row 1, give me mu[1, comp[1], :]," so on and so forth for all batch rows simultaneously.
    - `.squeeze(1)` removes the leftover size-1 dimension from the gather, squeezing the shape from `(batch, 1, D)` to `(batch, D)`
- `chosen_sigma = sigma_t.gather(1, idx).squeeze(1)` : chooses the `sigma` (processed standard deviation) to use
    - same process as the top
- `eps = torch.randn_like(chosen_mu)` : creates a tensor with same shape as `chosen_mu` with random numbers from a normal distribution of mean = 0 and var = 1
    - this is used to obtain a normal distribution of noise values, but with the same number of values as what is inside `chosen_mu`
- `z_next = chosen_mu + eps * chosen_sigma` : creates the distribution
    - `eps * chosen_sigma` does elementwise multiplication to scale the distribution
    - `chosen_mu + ` shifts the value 
    - returns `z_next` as the randomly sampled next state vector


##### 5.23 `mdn_loss()`
- calculate the loss value which is being minimized during training
- what train_mdnrnn backpropagates through
- ie, given the model's predicted mixture, how surprised should it by by the actual observed next state
- returns the negative log-likelihood of `target` under the predicted mixture.
- if the true target lands right on top of a high-weight, confident (small sigma) gaussian, loss is low.
- if the true target lands far from every predicted component, or the model was overconfident about the wrong place, loss is high.

```python
@staticmethod
def mdn_loss(pi, mu, sigma, target):
    target = target.unsqueeze(2)  
    log_prob = -0.5 * (((target - mu) / sigma) ** 2 + 2 * torch.log(sigma) + np.log(2 * np.pi))
    log_prob = log_prob.sum(dim=-1)  # (batch, seq_len, K)
    log_pi = torch.log(pi.clamp(min=1e-8))
    log_mix = torch.logsumexp(log_pi + log_prob, dim=-1)  # (batch, seq_len)
    return -log_mix.mean()
```

- `target = target.unsqueeze(2)` : adds a dimension so target with shape `(batch, seq_len, D)` can be compared against mu/sigma with shape `(batch, seq_len, K, D)` via broadcasting
    - i.e., compare the one true value against all K predicted components simultaneously
- `log_prob = -0.5 * (((target - mu) / sigma) ** 2 + 2 * torch.log(sigma) + np.log(2 * np.pi))` : the closed-form log-density of a gaussian
    - computing `-0.5 * ((x-mu)/sigma)^2 - log(sigma) - 0.5*log(2π)` per-dimension, 
- `log_prob = log_prob.sum(dim=-1)` : adds up the log-densities across the D dimensions
    - since the dimensions are treated as independent, multiplying probabilities = adding log-probabilities
- `log_pi = torch.log(pi.clamp(min=1e-8))` : applies `log()` and implements a floor of `1e-8`
- `log_mix = torch.logsumexp(log_pi + log_prob, dim=-1)` : combines "how likely is the target under component k" with "how much do we trust component k," summed 
    - `logsumexp()` is the numerically stable way to compute log(sum(exp(...))) over all K components 
    - this is the log-likelihood of the target under the whole mixture, not just one component.
- `return -log_mix.mean()` : returns the negative of this likelihood to change the maximisation problem into a minimisation problem 

##### 5.24 `init_hidden()`
- intitialises a pair of all-zero tensors shaped the way `nn.LSTM` expects its initial hidden/cell state: `(num_layers, batch_size, hidden_size)`

```python
def init_hidden(self, batch_size, device):
    h0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
    c0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
    return (h0, c0)
```

